In [1]:
import pandas as pd

fake_df = pd.read_csv("Fake.csv")
true_df = pd.read_csv("True.csv")

print("Fake shape:", fake_df.shape)
print("True shape:", true_df.shape)

fake_df.head()

Fake shape: (23481, 4)
True shape: (21417, 4)


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [2]:
fake_df["label"] = 0  # 0 = fake
true_df["label"] = 1  # 1 = real

df = pd.concat([fake_df, true_df], axis=0, ignore_index=True)

# shuffle so fake/real aren't in separate blocks
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
df["label"].value_counts()

(44898, 5)


label
0    23481
1    21417
Name: count, dtype: int64

In [3]:
# Check how often "Reuters" appears in real vs fake articles
df[df["label"] == 1]["text"].str.contains("Reuters").mean()

np.float64(0.9981790166690012)

In [4]:
df[df["label"] == 0]["text"].str.contains("Reuters").mean()

np.float64(0.013244751075337506)

In [5]:
import re

def strip_source_tag(text):
    # Removes patterns like "WASHINGTON (Reuters) - " from the start of text
    return re.sub(r'^.*?\(Reuters\)\s*-\s*', '', text)

df["text"] = df["text"].apply(strip_source_tag)

# Re-check leakage after cleaning
print("Real articles still mentioning Reuters:", df[df["label"] == 1]["text"].str.contains("Reuters").mean())
print("Fake articles still mentioning Reuters:", df[df["label"] == 0]["text"].str.contains("Reuters").mean())

Real articles still mentioning Reuters: 0.2302843535509175
Fake articles still mentioning Reuters: 0.013244751075337506


In [6]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 35918
Test size: 8980


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,   # keep only the 10,000 most informative words
    stop_words="english", # remove common filler words (the, is, and, etc.)
    ngram_range=(1, 2)    # consider single words AND two-word phrases
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (35918, 10000)
Test TF-IDF shape: (8980, 10000)


In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("Training complete")

Training complete


In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9801781737193764

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.98      4696
           1       0.97      0.99      0.98      4284

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980


Confusion Matrix:
 [[4582  114]
 [  64 4220]]


In [10]:
import joblib

joblib.dump(model, "baseline_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Saved model and vectorizer")

Saved model and vectorizer
